# Test Result

Dieses Notebook trainiert das Modell fuer mehrere Spieler jeweils auf genau `100` Trainingsspielen, evaluiert danach auf dem gehaltenen Testsplit und fasst die Ergebnisse ueber alle Spieler zusammen. Der Fokus liegt auf einem robusten Gesamtwert und auf dem Elo-vs-Top1-Plot.

In [ ]:
import importlib
import sys
from pathlib import Path

for helper_dir in (Path.cwd(), Path.cwd() / "notebooks"):
    if (helper_dir / "test_result_utils.py").exists() and str(helper_dir) not in sys.path:
        sys.path.insert(0, str(helper_dir))

import test_result_utils

importlib.reload(test_result_utils)
from test_result_utils import ExperimentConfig, generate_posthoc_phase_probability_plots, load_existing_train_games_comparison, resume_fixed_train_games_comparison, run_fixed_train_games_comparison

CONFIG = ExperimentConfig(
    player_usernames=(
        "Vlad_Lazarev79",
        "chesspawnrookking",
        "Art-Vega",
        "BigSabee",
        "iCe_eNerGyTeaM",
        "shaheus",
        "Padurean_Daniel",
        "Bhavya33",
        "dampooo",
        "rascarcapac87",
        "DubNik",
        "casomir",
        "borovicka",
        "VictorMih68",
        "EduardoQuinto357",
        "ravijay",
        "IPersist",
        "BVladimir1957",
        "Slavikan777",
        "saltinok",
        "untaltal",
        "PROChessPlAyer81",
        "Alex_litl",
        "Andressing",
        "Arzikul",
        "XEVANEX",
        "vx567q",
        "jiaxu",
        "Hefezopf22",
        "AnaSantos",
        "pradeep_68",
        "volod_magmel",
        "Vlr_Kharchuk",
        "Scalgero",
        "anwuman",
        "yuriy1936",
        "rexhademi1973",
        "Octavio59",
        "pandu_a",
        "palencia",
        "Tschucklo",
        "NoelRoc",
        "gino68",
        "Adolf76",
        "O__3axapoB__IIIaxTbI",
        "DonKaspadar",
        "burdano",
        "PetrusPaulus",
        "andalucianoise",
        "Aabdalla2012",
    ),
    train_games_per_player=500,
    max_games=900,
    perf_type="classical",
    split_strategy="chronological",
    test_frac=0.1,
    val_frac_within_train=0.05,
    min_context_ply=10,
    learning_rate=2.5e-4,

    num_train_epochs=1,
    lora_rank=32,
    lora_alpha=64,
    target_modules="all-linear",
    require_train_games_per_player=False,
    results_root_name="results",
)

RESULTS_DIR_TO_LOAD = ""
RESULTS_DIR_TO_RESUME = ""

CONFIG


In [ ]:
if RESULTS_DIR_TO_RESUME:
    RESULT = resume_fixed_train_games_comparison(RESULTS_DIR_TO_RESUME)
elif RESULTS_DIR_TO_LOAD:
    RESULT = load_existing_train_games_comparison(RESULTS_DIR_TO_LOAD)
else:
    RESULT = run_fixed_train_games_comparison(CONFIG)
SUMMARY = RESULT["comparison_summary"]
AGGREGATE = RESULT["aggregate_metrics"]

print("Results dir:", RESULT["results_dir"])
print("Successful players:", len(RESULT["successful_rows"]))
print("Failed players:", len(RESULT["failed_rows"]))

for top_k in (1, 3, 5, 10):
    print(f"Mean baseline top{top_k}:", round(AGGREGATE.get(f"mean_baseline_top{top_k}_accuracy", 0.0), 4))
    print(f"Mean finetuned top{top_k}:", round(AGGREGATE.get(f"mean_finetuned_top{top_k}_accuracy", 0.0), 4))
    print(f"Mean delta top{top_k}:", round(AGGREGATE.get(f"mean_delta_top{top_k}_accuracy", 0.0), 4))
    print(f"Weighted baseline top{top_k}:", round(AGGREGATE.get(f"weighted_baseline_top{top_k}_accuracy", 0.0), 4))
    print(f"Weighted finetuned top{top_k}:", round(AGGREGATE.get(f"weighted_finetuned_top{top_k}_accuracy", 0.0), 4))
    print(f"Weighted delta top{top_k}:", round(AGGREGATE.get(f"weighted_delta_top{top_k}_accuracy", 0.0), 4))
    print()

print("Mean baseline rank:", round(AGGREGATE.get("mean_baseline_mean_rank", 0.0), 4))
print("Mean finetuned rank:", round(AGGREGATE.get("mean_finetuned_mean_rank", 0.0), 4))
print("Mean delta rank:", round(AGGREGATE.get("mean_delta_mean_rank", 0.0), 4))
print("Mean KL divergence (finetuned vs baseline):", round(AGGREGATE.get("mean_average_kl_finetuned_vs_baseline", 0.0), 6))

SUMMARY


In [ ]:
try:
    import pandas as pd
    from IPython.display import display

    columns = [
        "username",
        "elo",
        "train_games_used",
        "test_examples",
        "baseline_top1_accuracy",
        "finetuned_top1_accuracy",
        "delta_top1_accuracy",
        "baseline_mean_rank",
        "finetuned_mean_rank",
    ]
    df = pd.DataFrame(RESULT["successful_rows"])
    if not df.empty:
        display(df[columns].sort_values(["elo", "username"], na_position="last").reset_index(drop=True))
    else:
        print("No successful rows available.")
except Exception as exc:
    print("Could not render DataFrame:", exc)
    RESULT["successful_rows"]


In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_paths = SUMMARY.get("plots", {})
for key in ["elo_vs_top1_accuracy", "elo_vs_delta_top1_accuracy", "players_sorted_by_elo_top1_accuracy", "training_loss_by_player", "training_loss_mean", "training_loss_quality_100_games", "phase_probability_baseline", "phase_probability_finetuned", "phase_probability_delta", "phase_top1_accuracy_by_move"]:
    plot_path = plot_paths.get(key)
    if plot_path and Path(plot_path).exists():
        print(key, plot_path)
        display(Image(filename=plot_path))
